# Andrea Pezzolla & Matteo Maruca
Progetto 2 DataScience, SUPSI, 2026

In [ ]:
# Import
import numpy as np
import pandas as pd
import plotly.express as px
from plotly.subplots import make_subplots
import plotly.graph_objects as go

In [ ]:
# Dataframe
df = pd.read_csv("100097.csv", sep=';')

In [ ]:
# TRADUZIONI COLONNE 

# ── Rinomina colonne in italiano ──────────────────────────────
df = df.rename(columns={
    "Timestamp"                         : "timestamp",
    "Messung-ID"                        : "id_postazione",
    "Richtung ID"                       : "id_direzione",
    "Geschwindigkeit"                   : "velocita_kmh",
    "Zeit"                              : "ora",
    "Datum"                             : "data",
    "Datum und Zeit"                    : "data_ora",
    "Messbeginn"                        : "inizio_misurazione",
    "Messende"                          : "fine_misurazione",
    "Zone"                              : "zona_limite",
    "Ort"                               : "localita",
    "Richtung"                          : "direzione",
    "Koordinaten"                       : "coordinate",
    "Übertretungsquote"                 : "tasso_infrazioni_pct",
    "Geschwindigkeit V50"               : "v50",
    "Geschwindigkeit V85"               : "v85",
    "Strasse"                           : "via",
    "Hausnummer"                        : "numero_civico",
    "Fahrzeuge"                         : "n_veicoli",
    "Fahrzeuglänge"                     : "lunghezza_veicolo_m",
    "Kennzahlen pro Mess-Standort"      : "link_statistiche",
    "geo_point_2d"                      : "geo_point",
})

# ── Conversioni di tipo ───────────────────────────────────────
df["timestamp"]           = pd.to_datetime(df["timestamp"], utc=True)
df["inizio_misurazione"]  = pd.to_datetime(df["inizio_misurazione"])
df["fine_misurazione"]    = pd.to_datetime(df["fine_misurazione"])

# Estrai colonne utili da timestamp
df["anno"]         = df["timestamp"].dt.year
df["mese"]         = df["timestamp"].dt.month
df["giorno_sett"]  = df["timestamp"].dt.day_name(locale="it_IT.UTF-8")  # es. "Lunedì"
df["ora_intera"]   = df["timestamp"].dt.hour

# Zona limite come categoria
#df["zona_limite"] = df["zona_limite"].astype(int).astype(str).apply(lambda x: f"Zona {x}")

print("Colonne rinominate:")
print(df.columns.tolist())
print(f"\nDimensioni: {df.shape[0]:,} righe × {df.shape[1]} colonne")
print(df.dtypes)

In [ ]:
df["zona_limite"].unique().max()

In [ ]:
# Distribuzione delle Velocità per Zona Limite

fig1 = px.histogram(
    df,
    x="velocita_kmh",
    color="zona_limite",
    nbins=60,
    barmode="overlay",
    opacity=0.75,
    color_discrete_map={"Zona 30": "#00B4D8", "Zona 50": "#E07B39"},
    title="Distribuzione delle Velocità per Zona Limite",
    labels={
        "velocita_kmh": "Velocità (km/h)",
        "count":        "Numero misurazioni",
        "zona_limite":  "Zona"
    }
)
fig1.show()

In [ ]:
# ── ANALISI 2 ────────────────────────────────────────────────
# Top 15 postazioni per tasso medio di infrazioni
# ────────────────────────────────────────────────────────────

# Aggiungiamo 'zona_limite' al groupby per conservare il dato
top_infrazioni = (
    df.groupby(["localita", "via", "zona_limite"])["tasso_infrazioni_pct"]
    .mean()
    .reset_index()
    .sort_values("tasso_infrazioni_pct", ascending=False)
    .head(15)
)

fig2 = px.bar(
    top_infrazioni,
    x="tasso_infrazioni_pct",
    y="via",
    orientation="h",
    color="tasso_infrazioni_pct",
    color_continuous_scale="Teal", 
    title="<b>Top 15 Postazioni per Tasso di Infrazioni</b>",
    labels={
        "tasso_infrazioni_pct": "Tasso infrazioni (%)",
        "via":                  "Via",
        "localita":             "Località",
        "zona_limite":          "Limite (km/h)"
    },
    # Configurazione dettagliata dell'hover
    hover_data={
        "via": False, # Nasconde la via dal box perché è già sull'asse Y
        "localita": True,
        "zona_limite": True,
        "tasso_infrazioni_pct": ":.2f" # Formatta il numero con 2 decimali (es. 12.34 al posto di 12.34567)
    }
)

fig2.update_layout(
    yaxis={"categoryorder": "total ascending"},
    coloraxis_showscale=False,
    plot_bgcolor="white", # Sfondo bianco per una lettura più pulita
)

# Aggiungiamo la griglia verticale leggera per facilitare la lettura dei valori
fig2.update_xaxes(showgrid=True, gridcolor="#E5E5E5")

fig2.show()


In [ ]:
# Zoom sulla peggiore postazione: Distribuzione velocità

# 1. Estraiamo automaticamente il nome della via e il limite dal primo in classifica
via_peggiore = top_infrazioni.iloc[0]["via"]
limite_peggiore = top_infrazioni.iloc[0]["zona_limite"]

# 2. Filtriamo il dataframe originale 'df' per prendere solo i dati di quella via
df_peggiore = df[df["via"] == via_peggiore]

# 3. Creiamo l'istogramma (la "campana") delle velocità misurate
fig3 = px.histogram(
    df_peggiore,
    x="velocita_kmh",
    nbins=30, # Regola questo numero per fare le barre più o meno spesse
    title=f"<b>Distribuzione delle Velocità - {via_peggiore}</b>",
    labels={
        "velocita_kmh": "Velocità Misurata (km/h)",
        "count": "Numero di Veicoli"
    },
    color_discrete_sequence=["#1f77b4"],
    opacity=0.8
)

# 4. Aggiungiamo la riga verticale rossa per indicare dove inizia l'infrazione
fig3.add_vline(
    x=limite_peggiore, 
    line_width=3, 
    line_dash="dash", 
    line_color="red",
    annotation_text=f"Limite ({limite_peggiore} km/h)", 
    annotation_position="top right",
    annotation_font_color="red",
    annotation_font_size=14
)

# 5. Pulizia del layout
fig3.update_layout(
    plot_bgcolor="white",
    bargap=0.05, # Piccolo spazio tra le barre per renderle più leggibili
    yaxis_title="Numero di Veicoli"
)

# Aggiungiamo griglie leggere
fig3.update_xaxes(showgrid=True, gridcolor="#E5E5E5")
fig3.update_yaxes(showgrid=True, gridcolor="#E5E5E5")

fig3.show()

In [ ]:
# Mappa di calore – Velocità media per giorno e ora

# Ordine corretto dei giorni
ordine_giorni = ["Lunedì", "Martedì", "Mercoledì", "Giovedì", "Venerdì", "Sabato", "Domenica"]

pivot = (
    df.groupby(["giorno_sett", "ora_intera"])["velocita_kmh"]
    .mean()
    .reset_index()
    .pivot(index="giorno_sett", columns="ora_intera", values="velocita_kmh")
    .reindex(ordine_giorni)
)

fig3 = px.imshow(
    pivot,
    color_continuous_scale="inferno",
    aspect="auto",
    title="<b>Velocità Media per Giorno della Settimana e Ora</b>",
    labels={
        "x":     "Ora del giorno",
        "y":     "Giorno",
        "color": "Velocità media (km/h)"
    }
)

# Aggiornamento dei font e del design generale
fig3.update_layout(
    title=dict(font=dict(size=20)),
    xaxis_title="<b>Ora del giorno</b>",
    yaxis_title="",
    plot_bgcolor="white",
    paper_bgcolor="white",
)

# Forza la visualizzazione di tutte le ore (da 0 a 23)
fig3.update_xaxes(
    tickmode="linear", 
    tick0=0, 
    dtick=1,
    showgrid=False # Rimuoviamo la griglia per non sporcare la mappa di calore
)

fig3.update_yaxes(showgrid=False)

# Tooltip personalizzato per una lettura più pulita
fig3.update_traces(
    hovertemplate="<b>%{y}</b> alle ore <b>%{x}:00</b><br>Velocità media: %{z:.2f} km/h<extra></extra>"
)

fig3.show()


In [ ]:
#Tasso Infrazioni per Giorno e Ora - Heatmap

df["infrazioni_assolute"] = (df["tasso_infrazioni_pct"] / 100) * df["n_veicoli"]
df["infrazioni_assolute"] = df["infrazioni_assolute"].fillna(0)

df_raggruppato = df.groupby(["giorno_sett", "ora_intera"])[["infrazioni_assolute", "n_veicoli"]].sum().reset_index()

df_raggruppato["tasso_medio_reale_pct"] = (df_raggruppato["infrazioni_assolute"] / df_raggruppato["n_veicoli"]) * 100

# Ordine corretto dei giorni
ordine_giorni = ["Lunedì", "Martedì", "Mercoledì", "Giovedì", "Venerdì", "Sabato", "Domenica"]

pivot_inf = (
    df_raggruppato.pivot(index="giorno_sett", columns="ora_intera", values="tasso_medio_reale_pct")
    .reindex(ordine_giorni)
)

fig4 = px.imshow(
    pivot_inf,
    color_continuous_scale="inferno",
    aspect="auto",
    title="<b>Tasso Infrazioni per Giorno e Ora (%)</b><br><sup style='font-size:13px; font-weight:normal; color:gray'>Calcolo del tasso ponderato sui volumi di traffico effettivi</sup>",
    labels={
        "x":     "Ora del giorno",
        "y":     "Giorno",
        "color": "Infrazioni (%)"
    }
)

fig4.update_layout(
    title=dict(font=dict(size=20)),
    xaxis_title="<b>Ora del giorno</b>",
    yaxis_title="",
    plot_bgcolor="white",
    paper_bgcolor="white",
)

# Forza la visualizzazione di tutte le ore (da 0 a 23)
fig4.update_xaxes(
    tickmode="linear", 
    tick0=0, 
    dtick=1,
    showgrid=False
)

fig4.update_yaxes(showgrid=False)

fig4.update_traces(
    hovertemplate="<b>%{y}</b> alle ore <b>%{x}:00</b><br>Tasso infrazioni: %{z:.2f}%<extra></extra>"
)

fig4.show()

In [ ]:
# Scatter V50 vs V85 per postazione

agg_postazioni = (
    df.groupby(["id_postazione", "via", "zona_limite"])
    .agg(
        v50=("v50",             "mean"),
        v85=("v85",             "mean"),
        infrazioni=("tasso_infrazioni_pct", "mean"),
        n_veicoli=("n_veicoli",     "mean")
    )
    .reset_index()
)

fig5 = px.scatter(
    agg_postazioni,
    x="v50",
    y="v85",
    color="zona_limite",
    size="infrazioni",
    hover_data=["via", "infrazioni", "n_veicoli"],
    color_discrete_map={"Zona 30": "#00B4D8", "Zona 50": "#E07B39"},
    title="Correlazione V50 – V85 per Postazione",
    labels={
        "v50":        "V50 – velocità mediana (km/h)",
        "v85":        "V85 – 85° percentile (km/h)",
        "zona_limite":"Zona",
        "infrazioni": "Tasso infrazioni (%)"
    }
)
fig5.show()

In [ ]:
# Distribuzione velocità per tipo di veicolo (lunghezza)
# Boxplot – velocita_kmh per categoria lunghezza veicolo

# Categorizza la lunghezza del veicolo
bins   = [0, 3.5, 6, 10, 100]
labels = ["Auto (<3.5m)", "Furgone (3.5–6m)", "Camion (6–10m)", "Autoarticolato (>10m)"]

df["cat_veicolo"] = pd.cut(
    df["lunghezza_veicolo_m"],
    bins=bins,
    labels=labels,
    right=True
)

fig6 = px.box(
    df,
    x="cat_veicolo",
    y="velocita_kmh",
    color="zona_limite",
    color_discrete_map={"Zona 30": "#00B4D8", "Zona 50": "#E07B39"},
    title="Distribuzione Velocità per Categoria di Veicolo e Zona",
    labels={
        "cat_veicolo":  "Categoria veicolo",
        "velocita_kmh": "Velocità (km/h)",
        "zona_limite":  "Zona"
    }
)
fig6.show()

In [ ]:
# Trend mensile – velocità media e infrazioni nel tempo
# Line chart – evoluzione mese per mese

# Aggregazione dati
trend = (
    df.groupby(["anno", "mese"])
    .agg(
        vel_media  = ("velocita_kmh", "mean"),
        infrazioni = ("tasso_infrazioni_pct", "mean")
    )
    .reset_index()
)
 
# FIX: Rinomina le colonne in inglese per farle capire a to_datetime
trend["periodo"] = pd.to_datetime(
    trend[["anno", "mese"]]
    .rename(columns={"anno": "year", "mese": "month"})
    .assign(day=1)
)
 
# Creazione Subplots
fig7 = make_subplots(
    rows=2, cols=1,
    shared_xaxes=True,
    vertical_spacing=0.1, # Leggero spazio in più tra i due grafici
    subplot_titles=("Velocità media (km/h)", "Tasso infrazioni medio (%)")
)

# Traccia 1: Velocità
fig7.add_trace(
    go.Scatter(
        x=trend["periodo"], 
        y=trend["vel_media"],
        mode="lines+markers", 
        name="Velocità",
        line=dict(color="#00B4D8", width=3),
        marker=dict(size=8, symbol="circle", line=dict(color="white", width=1)),
        hovertemplate="<b>%{x|%b %Y}</b><br>Velocità media: %{y:.2f} km/h<extra></extra>"
    ),
    row=1, col=1
)

# Traccia 2: Infrazioni
fig7.add_trace(
    go.Scatter(
        x=trend["periodo"], 
        y=trend["infrazioni"],
        mode="lines+markers", 
        name="Infrazioni",
        line=dict(color="#E07B39", width=3),
        marker=dict(size=8, symbol="circle", line=dict(color="white", width=1)),
        hovertemplate="<b>%{x|%b %Y}</b><br>Tasso infrazioni: %{y:.2f}%<extra></extra>"
    ),
    row=2, col=1
)

# Aggiornamento Layout per un look più pulito e professionale
fig7.update_layout(
    title=dict(text="<b>Trend Mensile</b> – Velocità Media e Tasso Infrazioni", font=dict(size=20)),
    plot_bgcolor="white",
    paper_bgcolor="white",
    showlegend=False,
    hovermode="x unified", # Mostra il tooltip per entrambi i grafici contemporaneamente
    height=600 
)

# Formattazione degli assi e della griglia
fig7.update_yaxes(title_text="km/h", row=1, col=1, showgrid=True, gridcolor="#E5E5E5")
fig7.update_yaxes(title_text="%", row=2, col=1, showgrid=True, gridcolor="#E5E5E5")
fig7.update_xaxes(showgrid=True, gridcolor="#E5E5E5", row=1, col=1)
fig7.update_xaxes(title_text="Mese", showgrid=True, gridcolor="#E5E5E5", row=2, col=1)

fig7.show()

In [ ]:
# Mappa interattiva delle postazioni (Plotly mapbox)

df[["lat", "lon"]] = df["geo_point"].str.split(",", expand=True).astype(float)

df["infrazioni_assolute"] = (df["tasso_infrazioni_pct"] / 100) * df["n_veicoli"]
df["infrazioni_assolute"] = df["infrazioni_assolute"].fillna(0)

mappa = (
    df.groupby(["id_postazione", "via", "zona_limite", "lat", "lon"])
    .agg(
        tot_infrazioni = ("infrazioni_assolute", "sum"),
        tot_veicoli    = ("n_veicoli", "sum"),
        vel_media      = ("velocita_kmh", "mean"),
        n_rilevazioni  = ("n_veicoli", "count") # Conta quante righe (misurazioni) abbiamo per questo radar
    )
    .reset_index()
)
mappa["infrazioni_pct"] = (mappa["tot_infrazioni"] / mappa["tot_veicoli"]) * 100

# Normalizziamo il volume (Grandezza pallino) per non penalizzare i radar accesi per meno tempo
mappa["veicoli_medi"] = mappa["tot_veicoli"] / mappa["n_rilevazioni"]

fig8 = px.scatter_map(
    mappa,
    lat="lat",
    lon="lon",
    color="infrazioni_pct",
    size="veicoli_medi",
    size_max=25, # Aumentato leggermente per rendere più visibili i pallini
    hover_name="via",
    hover_data={
        "lat": False,
        "lon": False,
        "infrazioni_pct": ":.2f", 
        "vel_media": ":.1f", 
        "zona_limite": True,
        "veicoli_medi": ":.0f"
    },
    color_continuous_scale="inferno",
    zoom=12,
    map_style="carto-positron",
    title="<b>Mappa Postazioni – Tasso di Infrazioni vs Volume di Traffico</b>",
    labels={
        "infrazioni_pct": "Infrazioni (%)",
        "vel_media":      "Vel. media (km/h)",
        "veicoli_medi":   "Veicoli (media/ril.)",
        "zona_limite":    "Limite"
    }
)

fig8.update_layout(
    margin=dict(l=0, r=0, t=50, b=0),
    title=dict(font=dict(size=20))
)

fig8.show()

# Grafici Maru

## Grafico scatter geo che mostra l'omogeneità delle registrazioni nel canton basilea

In [ ]:
df[["lat", "lon"]] = (
    df["geo_point"]
    .str.split(",", expand=True)
    .astype(float)
)

#df['zona_limite'] = df['zona_limite'].str.extract(r'(\d+)')[0].astype(int)

df['Superamento'] = df['velocita_kmh'] - df['zona_limite']
idx_max = df.groupby(['lat', 'lon'])['Superamento'].idxmax()
df_peggiori = df.loc[idx_max].reset_index(drop=True)

fig = px.scatter_map(
    df_peggiori,
    lat="lat",
    lon="lon",
    zoom=12,                                 # Livello di zoom ideale per una città/cantone
    hover_name='zona_limite',
    center={"lat": 47.5596, "lon": 7.5886},  # Coordinate del centro di Basilea
    title="Misurazioni di Velocità - superamento limite kmh",
    size="Superamento",
    color="velocita_kmh",
    color_continuous_scale="Reds"
)

fig.update_layout(map_style="carto-positron")

fig.show()

IL grafico mostra il superamento massimo registrato da ogni radar e si vede bene perche con il colore vediamo la velocita a cui andava e invece con la grandezza quanto ha superato il limite.

**TODO**
Mappa interattiva 
Trend temporale 
Correlazione meteo 
Differenza velocità – tipo di veicolo (moto/camion)


## Numero superamenti per giorno della settimana

In [ ]:
df['superamento'] = df['velocita_kmh'] - df['zona_limite']

# Uso .copy() per evitare il SettingWithCopyWarning di Pandas
df_superamento = df[df['superamento'] > 0].copy()

df_superamento['data'] = pd.to_datetime(df_superamento['data'], format='%d.%m.%y', errors='coerce')

# Mappatura dal numero del giorno al nome in italiano
dizionario_giorni = {
    0: 'Lunedì', 
    1: 'Martedì', 
    2: 'Mercoledì', 
    3: 'Giovedì', 
    4: 'Venerdì', 
    5: 'Sabato', 
    6: 'Domenica'
}

# Estraiamo il giorno della settimana (da 0 a 6) e lo traduciamo con il dizionario
df_superamento['giorno_settimana'] = df_superamento['data'].dt.dayofweek.map(dizionario_giorni)

# Lista per forzare l'ordine logico (e non alfabetico) nel grafico
ordine_giorni_it = ['Lunedì', 'Martedì', 'Mercoledì', 'Giovedì', 'Venerdì', 'Sabato', 'Domenica']

fig = px.histogram(
    df_superamento,
    y='giorno_settimana',
    category_orders={"giorno_settimana": ordine_giorni_it},
    text_auto=True,
    title="Numero di infrazioni per giorno della settimana",
    labels={'giorno_settimana': 'Giorno della Settimana', 'count': 'Numero di Infrazioni'}
)

fig.show()

C'è una tendenza a fare infrazioni minore nel weekend rispetto al resto della settimana

In [ ]:
df['zona_limite'].unique()

In [ ]:
df[["lat", "lon"]] = (
    df["geo_point"]
    .str.split(",", expand=True)
    .astype(float)
)

df_notte_alto_limite = df[
    (df['ora_intera'].isin([22, 23, 0, 1, 2, 3, 4, 5])) & 
    (df['zona_limite'] >= 50) &
    (df['velocita_kmh'] > df['zona_limite'])
].copy()

df_notte_alto_limite['superamento'] = df_notte_alto_limite['velocita_kmh'] - df_notte_alto_limite['zona_limite']
idx_max = df_notte_alto_limite.groupby(['lat', 'lon'])['superamento'].idxmax()

df_peggiore_notte = df_notte_alto_limite.loc[idx_max]

fig = px.scatter_map(
    df_peggiore_notte,
    lat="lat",
    lon="lon",
    size="zona_limite",
    color="superamento",
    color_continuous_scale="Inferno",
    hover_name="superamento",
    hover_data=["velocita_kmh", "zona_limite", "ora", "data"],
    zoom=12,
    center={"lat": 47.5596, "lon": 7.5886},
    map_style="carto-positron",
    title="Pirati della Notte: Infrazioni su Strade ad Alto Limite (>= 50 km/h, 22:00 - 05:00)"
)

fig.update_layout(margin={"r":0, "t":40, "l":0, "b":0})

fig.show()